In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [18]:
class SingleHeadAttention(nn.Module):
    def __init__(self, d_model, d_head):
        super().__init__()
        self.d_head = d_head
        self.W_q = nn.Linear(d_model, d_head, bias = False)
        self.W_k = nn.Linear(d_model, d_head, bias = False)
        self.W_v = nn.Linear(d_model, d_head, bias = False)

    def forward(self, x):
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        scores = (1/math.sqrt(self.d_head))*(Q @ K.T)

        seq_len = x.shape[0]
        mask = torch.ones((seq_len, seq_len))
        causal_mask = torch.triu(mask, diagonal= 1)
        scores = scores.masked_fill(causal_mask.bool(), float('-inf'))

        attn_weights = torch.softmax(scores, dim = 1)
        # print(attn_weights.shape)
        # print(attn_weights)
        output = attn_weights @ V

        return output

In [19]:
torch.manual_seed(0)
d_model, d_head, seq_len = 8, 4, 5

attn = SingleHeadAttention(d_model, d_head)
x = torch.rand(seq_len, d_model)

output = attn(x)
print(output.shape)
print(output)

torch.Size([5, 4])
tensor([[ 0.5090,  0.1187, -0.1649,  0.1992],
        [ 0.3121,  0.0163, -0.0886,  0.0256],
        [ 0.2261,  0.1202,  0.0331, -0.0537],
        [ 0.1913,  0.2108, -0.0530, -0.0476],
        [ 0.1604,  0.2039, -0.1089, -0.0579]], grad_fn=<MmBackward0>)


In [20]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        d_head = d_model // num_heads
        self.heads = nn.ModuleList([
            SingleHeadAttention(d_model, d_head) for _ in range(num_heads)
        ])
        self.W_o = nn.Linear(d_model, d_model, bias = False)

    def forward(self, x):
        head_outputs = [head(x) for head in self.heads]
        output = torch.cat(head_outputs, dim = 1)
        output = self.W_o(output)
        return output

In [21]:
torch.manual_seed(0)
d_model, num_heads, seq_len = 8, 2, 5

mha = MultiHeadAttention(d_model, num_heads)
x = torch.rand(seq_len, d_model)

output = mha(x)
print(output.shape)  # should be (seq_len, d_model) = (5, 8)

torch.Size([5, 8])


In [22]:
class MLP(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias = False)
        self.up_proj = nn.Linear(d_model, d_ff, bias = False)
        self.down_proj = nn.Linear(d_ff, d_model, bias = False)

    def forward(self, x):
        gate = self.gate_proj(x)
        up = self.up_proj(x)
        fused = F.silu(gate) * up
        output = self.down_proj(fused)
        return output